# Crash Course on NumPy, pandas, matplotlib & seaborn for ML Workflows

This notebook is a condensed, practical reference for the operations from these four libraries that come up constantly in the **early stages of an ML workflow**: loading data, cleaning it, engineering simple features, and visualizing it before modeling.

This is **not** a comprehensive introduction — for that, see their respective documentations:

- numpy: https://numpy.org/doc/stable/
- pandas: https://pandas.pydata.org/docs/
- matplotlib: https://matplotlib.org/stable/contents.html
- seaborn: https://seaborn.pydata.org/

Think of this as a cheat-sheet you can work through quickly, applied end-to-end on one real dataset.

**Dataset:** California Housing (`data/housing.csv`) — 20,640 rows describing housing in California census block groups, with the goal (in later notebooks) of predicting `median_house_value`.

**What this notebook covers:**
1. NumPy essentials — vectorized math, masking, basic stats
2. pandas essentials — selection, missing values, feature creation, grouping
3. matplotlib essentials — fine-grained plot control
4. seaborn essentials — fast statistical plots

## 0. Setup

Each library has a conventional import alias — `np`, `pd`, `plt`, `sns` — that you'll see in virtually every ML codebase.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("train.csv")

In [ ]:
df.shape

Before doing anything else, get oriented: how many rows/columns, what types, any missing values, what the numbers look like.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 1. NumPy Essentials

pandas DataFrames are built on top of NumPy arrays, and most numeric work under the hood (in pandas, scikit-learn, etc.) is NumPy. Pulling a column out as a raw array and operating on it directly is common when you want fast vectorized math without DataFrame overhead.

In [ ]:
income = df["median_income"].to_numpy()
print(type(income), income.dtype, income.shape)
income[:5]

### Vectorized math

NumPy applies operations to every element at once — no Python loop needed. This is faster and is the basis for how feature scaling/transforms work in ML pipelines.

In [ ]:
# median_income is in tens of thousands of dollars; convert to dollars
income_dollars = income * 10_000
income_dollars[:5]

### Boolean masking

A comparison on an array produces a boolean array of the same shape, which can then be used to filter — this is the same mechanism pandas uses for row filtering.

In [ ]:
high_income_mask = income > 8
print("high earners:", high_income_mask.sum(), "out of", len(income))
income[high_income_mask][:5]

### `np.where` for conditional logic

A vectorized if/else — useful for quick binning or flagging without writing a loop.

In [ ]:
income_tier = np.where(income < 3, "low", np.where(income < 6, "mid", "high"))
np.unique(income_tier, return_counts=True)

### Basic statistics

`mean`, `median`, `std`, and `percentile` are the building blocks of almost every summary statistic and normalization step in ML.

In [ ]:
print("mean:  ", np.mean(income))
print("median:", np.median(income))
print("std:   ", np.std(income))
print("25/50/75th pct:", np.percentile(income, [25, 50, 75]))

**Takeaway:** reach for NumPy directly when you need fast vectorized math, masking, or transforms on raw numeric arrays — pandas wraps these same operations for labeled, tabular data.

## 2. pandas Essentials

pandas is where most real data cleaning and feature prep happens — selecting, filtering, handling missing data, and deriving new columns.

### Selecting rows and columns

`.loc` selects by label, `.iloc` selects by integer position. Both accept row and column selectors.

In [ ]:
df.loc[0:4, ["longitude", "latitude", "median_house_value"]]

In [ ]:
df.iloc[0:5, 0:3]

### Filtering rows

Boolean conditions on a column (or combination of columns, with `&`/`|`) filter the DataFrame — same idea as NumPy masking, but it keeps all related columns aligned.

In [ ]:
near_ocean_expensive = df[(df["ocean_proximity"] == "NEAR OCEAN") & (df["median_house_value"] > 400_000)]
near_ocean_expensive.shape

### Handling missing values

`total_bedrooms` has missing values — check first, then decide on a strategy. A common, simple choice is filling with the median (robust to outliers).

In [ ]:
df.isna().sum()[df.isna().sum() > 0]

In [ ]:
median_bedrooms = df["total_bedrooms"].median()
df["total_bedrooms"] = df["total_bedrooms"].fillna(median_bedrooms)
df["total_bedrooms"].isna().sum()

### Creating derived features

Raw counts like `total_rooms` and `households` are more useful as ratios — this is basic feature engineering, turning per-block totals into per-household/per-room rates.

In [ ]:
df["rooms_per_household"] = df["total_rooms"] / df["households"]
df["bedrooms_per_room"] = df["total_bedrooms"] / df["total_rooms"]
df["population_per_household"] = df["population"] / df["households"]

df[["rooms_per_household", "bedrooms_per_room", "population_per_household"]].describe()

### Grouping and aggregating

`groupby` splits the data by a category, applies an aggregation per group, and combines the results — essential for understanding how a target varies across categories.

In [ ]:
df.groupby("ocean_proximity")["median_house_value"].agg(["mean", "median", "count"]).sort_values("mean", ascending=False)

### Counting categories

`value_counts` is the fastest way to see the distribution of a categorical column.

In [ ]:
df["ocean_proximity"].value_counts()

**Takeaway:** pandas is for labeled, tabular work — selection, filtering, missing-value handling, feature derivation, and group-wise summaries. It calls down to NumPy for the actual math.

## 3. matplotlib Essentials

matplotlib gives full, explicit control over a plot — figure size, axes, labels, colors. seaborn (next section) builds on top of it for faster statistical plots, but matplotlib is what you reach for when you need precise customization.

### A basic histogram

`plt.subplots()` creates a figure and an axes object together — the standard pattern for any plot you plan to customize.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["median_house_value"], bins=50, color="steelblue", edgecolor="white")
ax.set_xlabel("Median House Value ($)")
ax.set_ylabel("Count")
ax.set_title("Distribution of Median House Value")
plt.show()

### A geographic scatter plot

This dataset's `longitude`/`latitude` columns trace the shape of California. Coloring each point by `median_house_value` turns a scatter plot into a map of price patterns — a classic first look at this dataset.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
scatter = ax.scatter(
    df["longitude"], df["latitude"],
    c=df["median_house_value"], cmap="viridis",
    alpha=0.4, s=10,
)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("California Housing Prices by Location")
fig.colorbar(scatter, ax=ax, label="Median House Value ($)")
plt.show()

### Multiple subplots

`plt.subplots(rows, cols)` returns a grid of axes, useful for comparing several features at a glance.

In [ ]:
features = ["housing_median_age",
            "median_income",
            "rooms_per_household",
            "population_per_household"]

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, feature in zip(axes.flat, features):
    ax.hist(df[feature], bins=40, color="darkorange", edgecolor="white")
    ax.set_title(feature)
    ax.set_xlabel(feature)
    ax.set_ylabel("Count")
fig.tight_layout()
plt.show()

**Takeaway:** matplotlib = full control. Use it when you need a specific layout, custom colormap, or fine-tuned annotation that a higher-level library doesn't expose.

## 4. seaborn Essentials

seaborn wraps matplotlib with sensible defaults and built-in statistical logic — fewer lines of code for common EDA plots like distributions, category comparisons, and correlations.

### Distribution plots

`histplot` (with an optional KDE curve) is seaborn's quick alternative to a matplotlib histogram.

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["median_house_value"], bins=50, kde=True, color="steelblue")
plt.title("Distribution of Median House Value")
plt.show()

### Comparing a numeric column across categories

`boxplot` shows the spread (and outliers) of a numeric column for each category in one call — much less code than building this manually.

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="ocean_proximity", y="median_house_value")
plt.title("Median House Value by Ocean Proximity")
plt.xticks(rotation=20)
plt.show()

### Correlation heatmap

`df.corr()` computes pairwise correlations between numeric columns; `sns.heatmap` visualizes the matrix. This is usually one of the first things to check before modeling — it flags which features move together (and might be redundant) and which correlate most with the target.

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
corr = numeric_df.corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
corr_masked = corr.mask(mask)

plt.figure(figsize=(9, 7))
sns.heatmap(corr_masked, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature Correlation Matrix")
plt.show()

### Pairwise relationships

`pairplot` plots every pair of chosen columns against each other, with histograms on the diagonal — a fast way to spot relationships and clusters. It's expensive to compute, so it's typically run on a sample and a small subset of columns rather than the full dataset.

In [ ]:
sample = df.sample(1000, random_state=42)
cols = ["median_income", "housing_median_age", "rooms_per_household", "median_house_value"]

sns.pairplot(sample[cols], diag_kind="kde", plot_kws={"alpha": 0.4, "s": 15})
plt.show()

**Takeaway:** seaborn = speed for statistical EDA. Reach for it first when exploring a new dataset; drop down to matplotlib when you need finer control over the result.